# ML Practical: Uber Fare Prediction

**Aim:** Predict Uber ride fare using Linear Regression and Random Forest Regression.

**Dataset:** `uber.csv`


### Before running
Keep this notebook in the same folder as its CSV dataset. Run the cells from top to bottom.

If a library is missing, run this in a code cell:
`%pip install pandas numpy scikit-learn matplotlib scipy`

After installation, restart the kernel if Jupyter asks you to.

In [ ]:
# ML Practical 7: Uber Fare Prediction
# Linear Regression and Random Forest Regression

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

DATA_FILE = "uber.csv"


In [ ]:
# 1. Load dataset
df = pd.read_csv(DATA_FILE)


In [ ]:
# 2. Keep required columns and remove missing values
columns = [
    "fare_amount", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "passenger_count"
]
df = df[columns].dropna()


In [ ]:
# 3. Remove invalid fares, passenger counts and corrupted GPS records
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] <= 250)]
df = df[df["passenger_count"].between(1, 6)]
df = df[
    df["pickup_longitude"].between(-75, -72)
    & df["dropoff_longitude"].between(-75, -72)
    & df["pickup_latitude"].between(40, 42)
    & df["dropoff_latitude"].between(40, 42)
]

if len(df) < 10:
    raise ValueError("Not enough valid Uber records after cleaning.")


In [ ]:
# 4. Separate input features and target
X = df.drop("fare_amount", axis=1)
y = df["fare_amount"]


In [ ]:
# 5. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
# 6. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print("LINEAR REGRESSION")
print("MSE:", round(mean_squared_error(y_test, lr_pred), 3))
print("R2 Score:", round(r2_score(y_test, lr_pred), 3))


In [ ]:
# 7. Random Forest Regression
rf = RandomForestRegressor(
    n_estimators=30, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("\nRANDOM FOREST REGRESSION")
print("MSE:", round(mean_squared_error(y_test, rf_pred), 3))
print("R2 Score:", round(r2_score(y_test, rf_pred), 3))


In [ ]:
# 8. Predict fare for a new ride
new_ride = pd.DataFrame([{
    "pickup_longitude": -73.99,
    "pickup_latitude": 40.73,
    "dropoff_longitude": -73.98,
    "dropoff_latitude": 40.75,
    "passenger_count": 2
}])

print("\nLinear Regression predicted fare:",
      round(lr.predict(new_ride)[0], 2))
print("Random Forest predicted fare:",
      round(rf.predict(new_ride)[0], 2))
